# Final Project: Black Friday Excel Data

In [1]:
import openpyxl as xl

# pprint prints dictionaries a bit more nicely than print
from pprint import pprint

## Part 1

First, read in maven_ski_shop_data.xlsx

Then, assign the orders worksheet to an intuitive variable name. 

In [2]:
read_wb = xl.load_workbook('maven_ski_shop_data.xlsx')

### Column Printer Function

Since we'll be working with Excel frequently, let's create a helper function to print all the rows a specified column.

This will help us view imported data without looking at Excel, and review the data we've written into columns without needing to save a file to review the changes.

Your function should take two arguments:

* Worksheet where data is located
* Column Letter of data to print

Your function should:

* Print Cell Coordinate (e.g. A1, D4).
* Print the contents of the cell.

Then call your function twice:

* Once to print Order ID (column A in 'Orders_Info')
* Once to print subtotal (column D in 'Orders_Info')

In [3]:
def column_printer(sheet, col_letter):
    col = []
    for cell in sheet[col_letter]:
        col.append(cell.value)
    return col

In [4]:
orders_sheet = read_wb['Orders_Info']

In [5]:
column_printer(orders_sheet, 'A')[:3]

['Order_ID', 100000, 100001]

In [6]:
column_printer(orders_sheet, 'D')[:3]

['Subtotal', 15.98, 899.97]

### Order Data Dictionary

Create a dictionary with all of the information contained in the 'Orders_Info' Worksheet.

* The dictionary keys should be Order IDs (Column A)
* The values should a list storing the data in the rest of the columns.
* The columns in the list stored as values should be B, C, D, G, and H (after converting to list). 
* For example: The first order, column H should be the nested list: [10001, 10002]


A few notes:

* Convert the Items_Ordered field into a list (we covered a helpful string method for this)
* This can be done with a dictionary comprehension
* DO NOT INCLUDE EXCEL COLUMN HEADERS

In [7]:
#create column variable for each column in the orders sheet.
#only get the value starting from the index 1 to skip the header row.
ids_col = column_printer(orders_sheet, 'A')[1:]
b_col = column_printer(orders_sheet, 'B')[1:]
c_col = column_printer(orders_sheet, 'C')[1:]
d_col = column_printer(orders_sheet, 'D')[1:]
g_col = column_printer(orders_sheet, 'G')[1:]
h_col = column_printer(orders_sheet, 'H')[1:]

#create a dictionary with the order_id as the key and a list of the other columns as the value.
#colum h is a string of comma separated values, so we need to split it into a list of integers.
order_dict = {
    order_id: [b,c,d,g, [int(item) for item in str(h).split(',')]] 
    for order_id, b, c, d, g, h in zip(ids_col, b_col, c_col, d_col, g_col, h_col)
}

pprint(list(order_dict.items())[:5])

[(100000, ['C00004', '11/26/2021', 15.98, 'Sun Valley', [10001, 10002]]),
 (100001, ['C00007', '11/26/2021', 899.97, 'Stowe', [10008, 10009, 10010]]),
 (100002, ['C00015', '11/26/2021', 799.97, 'Mammoth', [10011, 10012, 10013]]),
 (100003,
  ['C00016', '11/26/2021', 117.96, 'Stowe', [10002, 10003, 10004, 10006]]),
 (100004, ['C00020', '11/26/2021', 5.99, 'Sun Valley', [10001]])]


### Sales Tax Calculation (Last time!)

We need to calculate the sales tax and total amount owed for every order in this sheet.

* If location is Sun Valley, apply a sales tax of 8%
* If location is Mammoth, apply a sales tax of 7.75%.
* If location is Stowe, apply a sales tax of 6%

Use the tax_calculator function to apply sales tax to each subtotal.

**Insert** the calculated sales tax and total amounts into your customer dictionary immediately after subtotal.

**Tip**: Figure out how to do this once and then loop through the column.

In [8]:
import tax_calculator as tc
for key, value in order_dict.items():
    if value[3] == 'Sun Valley':
        tax_rate = tc.tax_calculator(value[2], 0.08)
    elif value[3] == 'Mammoth':
        tax_rate = tc.tax_calculator(value[2], 0.0775)
    else:
        tax_rate = tc.tax_calculator(value[2], 0.06)
    value.insert(3, tax_rate[1])
    value.insert(4, tax_rate[2])

pprint(list(order_dict.items())[:5])

[(100000,
  ['C00004', '11/26/2021', 15.98, 1.28, 17.26, 'Sun Valley', [10001, 10002]]),
 (100001,
  ['C00007',
   '11/26/2021',
   899.97,
   54.0,
   953.97,
   'Stowe',
   [10008, 10009, 10010]]),
 (100002,
  ['C00015',
   '11/26/2021',
   799.97,
   62.0,
   861.97,
   'Mammoth',
   [10011, 10012, 10013]]),
 (100003,
  ['C00016',
   '11/26/2021',
   117.96,
   7.08,
   125.04,
   'Stowe',
   [10002, 10003, 10004, 10006]]),
 (100004, ['C00020', '11/26/2021', 5.99, 0.48, 6.47, 'Sun Valley', [10001]])]


### Write Sales Tax and Total Into the Excel Sheet


Great job! Now we just need to write this data into the workbook.

Write the sales tax and total you just calculated into the workbook, then save!

Call this workbook 'maven_ski_shop_data_fixed'.

**Tip:** There are a few ways to do this. As always, be patient, solve one step at a time.

In [9]:
#0. Create a new wb
new_wb = xl.Workbook()
ws = new_wb.active
ws.title = 'Orders_Fixed'

#1. Add the headers to the new sheet
headers = [orders_sheet['A1'].value, orders_sheet['B1'].value, orders_sheet['C1'].value, 
           'Tax Rate', orders_sheet['D1'].value, orders_sheet['G1'].value, orders_sheet['H1'].value]
ws.append(headers)

#2. Add the values to the new sheet to it corresponding header column
for key, value in order_dict.items():
    #get all the column value except column H 
    row = [key] + value[:-1]
    #add column H value to the end of the row
    row.append(','.join([str(item) for item in value[-1]]))
    #add the rows to the new sheet
    ws.append(row)

#3. Save the workbook with the new sheet added
new_wb.save('maven_ski_shop_data_fixed.xlsx')

## Part 2

### Analysis Time!


Now that we've fixed the data - it's time to perform analysis on our sales.

Our starting point will be the order_dict we created, after we added the taxes and totals columns.

Our first step will be to write a function that calculates the sum of a 'column' of data in our dictionary.

A 'column' for example, would be subtotals, which is at index 2 in the list stored as our dictionary values.

Your function should take the following arguments:

* Column Index (the index of the data of interest in the list stored in our dictionary values)
* Dictionary Name

It should output:

* The sum of values in the column (rounded to two decimal places)

Assume only numeric values will be in the column (You can develop cleaning logic later :D)

**Tip:** Use a list comprehension to retrieve the values of interest.

In [10]:
def column_sum(col_index, dict_name):
    col_sum = 0
    for key, value in dict_name.items():
        col_sum += value[col_index]
    return round(col_sum, 2)

#### Sum The Subtotal, Tax, and Total Columns 

Now that we have our column sum function, calculate the sum of:

* Subtotals
* Taxes
* Totals

In [11]:
print(f"Sum Subtotals: {column_sum(2, order_dict)}")
print(f"Sum Taxes: {column_sum(3, order_dict)}")
print(f"Totals: {column_sum(4, order_dict)}")

Sum Subtotals: 8731.47
Sum Taxes: 617.2
Totals: 9348.67


### What is the average of our subtotals?

Calculate the average value of our transactions.

Remember that each entry in your dictionary is one order.

In [27]:
print(f"Average Subtotal: ${round(column_sum(4, order_dict)/len(order_dict), 2)}")

Average Subtotal: $346.25


### How many unique customers did we have?

Calculate the total number of unique customers in our sales data.

Then calculate the number of orders per customer (total orders / unique customers)

In [13]:
# Unique customers
unique_cust = len(set([value[0] for value in order_dict.values()]))

# Average N.O of orders per customer
order_per_cust = len(order_dict)/unique_cust
print(f"Unique Customers: {unique_cust}" )
print(f"Order number per customer: {round(order_per_cust, 2)}")

Unique Customers: 19
Order number per customer: 1.42


### How many items in total did we sell?

Calculate the total number of items we sold in across all orders. 

This information is in Column H, which should be the last element in order_dict's values.

In [14]:
total_items_per_order = 0
for value in order_dict.values():
    total_items_per_order += len(value[-1])
print(f"Total items per order: {total_items_per_order}")

Total items per order: 54


### Sales By Location

Calculate the sum of subtotals by location.

Create a dictionary to store them, where location is the key, and revenue for that location is the value.

A few steps to consider:

* Loop through your dictionary
* build a dictionary as you go with location as key
* increment revenue every time a transaction matches the location.


Your output should look like 
{'Location1': sum of subtotals for 'Location1'}

With an entry for each location.

In [15]:
pprint(list(order_dict.items())[:5])

[(100000,
  ['C00004', '11/26/2021', 15.98, 1.28, 17.26, 'Sun Valley', [10001, 10002]]),
 (100001,
  ['C00007',
   '11/26/2021',
   899.97,
   54.0,
   953.97,
   'Stowe',
   [10008, 10009, 10010]]),
 (100002,
  ['C00015',
   '11/26/2021',
   799.97,
   62.0,
   861.97,
   'Mammoth',
   [10011, 10012, 10013]]),
 (100003,
  ['C00016',
   '11/26/2021',
   117.96,
   7.08,
   125.04,
   'Stowe',
   [10002, 10003, 10004, 10006]]),
 (100004, ['C00020', '11/26/2021', 5.99, 0.48, 6.47, 'Sun Valley', [10001]])]


In [16]:
location_sales_sum = {}

for value in order_dict.values():
    if value[5] not in location_sales_sum:
        #create dictionary key and set the value to 0
        location_sales_sum[value[5]] = 0

    # Increment by it subtotal
    location_sales_sum[value[5]] += value[2]

location_sales_sum

{'Sun Valley': 1268.84,
 'Stowe': 3582.8199999999993,
 'Mammoth': 3879.8099999999995}

### Challenge: Aggregator Function

Now that you've summed revenue by category (you just summed by location), can you write a function to generalize calculating a sum of a column, grouped by the unique values in another column? (for example, sum of totals by date or customer_id).

Your function should take the following arguments:
* index of the 'column' (index position in order_dict) to group by
* index of the 'column' (index position in order_dict) to sum by category
* the dictionary where the data is located (assume the same structure as order_dict.

It should return:
* A dictionary with the categories as keys, and the sum by category as value.

Once you've done so use your function to sum totals by date and customer_id.

In [17]:
def aggregator(group_by_index, sum_index, dict_name):
    agg_dict = {}
    for value in dict_name.values():
        if value[group_by_index] not in agg_dict:
            #create dictionary key and set the value to 0
            agg_dict[value[group_by_index]] = 0

        # Increment by it subtotal
        agg_dict[value[group_by_index]] += value[sum_index]

    return agg_dict

In [18]:
# index 1 = date, index 4 = total
sales_by_date =aggregator(1, 4, order_dict)
pprint({k: round(v, 2) for k, v in sorted(sales_by_date.items())}) #round & sorted by date

{'11/26/2021': 5915.18, '11/27/2021': 1714.25, '11/28/2021': 1719.24}


In [19]:
# index 0 = customer_id
sales_by_customer =aggregator(0, 4, order_dict)
pprint({k: round(v, 2) for k, v in sorted(sales_by_customer.items())}) #round & sorted by customer_id

{'C00001': 2079.48,
 'C00002': 26.49,
 'C00003': 107.99,
 'C00004': 146.85,
 'C00005': 28.06,
 'C00006': 53.86,
 'C00007': 953.97,
 'C00008': 794.97,
 'C00010': 1077.46,
 'C00013': 97.19,
 'C00014': 275.35,
 'C00015': 861.97,
 'C00016': 220.43,
 'C00017': 635.99,
 'C00018': 1165.95,
 'C00019': 701.98,
 'C00020': 6.47,
 'C00021': 107.74,
 'C00022': 6.47}


In [24]:
# Finding top 3 customers by sales
sorted(sales_by_customer.items(), key=lambda x: x[1], reverse=True)[:3]

[('C00001', 2079.48), ('C00018', 1165.95), ('C00010', 1077.46)]

In [26]:
# Finding the top customer and their share of total sales
top_cust = max(sales_by_customer, key=sales_by_customer.get)
top_share = sales_by_customer[top_cust] / sum(sales_by_customer.values()) * 100
print(f"{top_cust} spent ${sales_by_customer[top_cust]:,.2f}, accounted for {top_share:.1f}% of total sales")


C00001 spent $2,079.48, accounted for 22.2% of total sales
